In [1]:
import polars as pl
import pandas as pd
import numpy as np
import catboost
import os
import gc
from datetime import date, timedelta


In [2]:
test_start_date = date(2024, 8, 1)
val_start_date = date(2024, 7, 1)
val_end_date = date(2024, 7, 31)
train_end_date = date(2024, 6, 30)
data_path = './predict-user-fresh-order'

# Read data

In [3]:
actions_history = pl.scan_parquet(os.path.join(data_path, 'actions_history'))
search_history = pl.scan_parquet(os.path.join(data_path, 'search_history'))
product_information = pl.scan_csv(os.path.join(data_path, 'product_information.csv'))

In [4]:
product_information.head(100)

In [5]:
pl.read_csv(os.path.join(data_path, 'action_type_info.csv'))

action_type,action_type_id
str,i64
"""click""",1
"""favorite""",2
"""order""",3
"""search""",4
"""to_cart""",5
"""view""",6


In [6]:
val_start_date, val_end_date

(datetime.date(2024, 7, 1), datetime.date(2024, 7, 31))

In [7]:
val_target = (
    actions_history
    .filter(pl.col('timestamp').dt.date() >= val_start_date)
    .filter(pl.col('timestamp').dt.date() <= val_end_date)
    .select('user_id', (pl.col('action_type_id') == 3).alias('has_order'))
    .group_by('user_id')
    .agg(pl.max('has_order').cast(pl.Int32).alias('target'))
)

In [8]:
val_target.group_by('target').agg(pl.count('user_id'))

In [9]:
val_target.group_by('target').agg(pl.count('user_id')).collect_schema()

Schema([('target', Int32), ('user_id', UInt32)])

# Simple pipeline

## Feats

In [10]:
train_end_date, train_end_date - timedelta(days=30 * 4)

(datetime.date(2024, 6, 30), datetime.date(2024, 3, 2))

In [11]:
actions_aggs = {}
actions_id_to_suf = {
    1: "click",
    2: "favorite",
    3: "order",
    5: "to_cart",
}
for id_, suf in actions_id_to_suf.items():
    aggs = (
        actions_history
        .filter(pl.col('timestamp').dt.date() <= train_end_date)
        .filter(pl.col('timestamp').dt.date() >= train_end_date - timedelta(days=30 * 4))
        .filter(pl.col('action_type_id') == id_)
        .join(
            product_information
            .select('product_id', 'discount_price'),
            on='product_id',
        )
        .group_by('user_id')
        .agg(
            pl.count('product_id').cast(pl.Int32).alias(f'num_products_{suf}'),
            pl.sum('discount_price').cast(pl.Float32).alias(f'sum_discount_price_{suf}'),
            pl.max('discount_price').cast(pl.Float32).alias(f'max_discount_price_{suf}'),
            pl.max('timestamp').alias(f'last_{suf}_time'),
            pl.min('timestamp').alias(f'first_{suf}_time'),
        )
        .with_columns([
            (pl.lit(val_start_date) - pl.col(f'last_{suf}_time'))
            .dt.total_days()
            .cast(pl.Int32)
            .alias(f'days_since_last_{suf}'),

            (pl.lit(val_start_date) - pl.col(f'first_{suf}_time'))
            .dt.total_days()
            .cast(pl.Int32)
            .alias(f'days_since_first_{suf}'),
        ])
        .select(
            'user_id',
            f'num_products_{suf}',
            f'sum_discount_price_{suf}',
            f'max_discount_price_{suf}',
            f'days_since_last_{suf}',
            f'days_since_first_{suf}',
        )
    )
    actions_aggs[id_] = aggs

In [12]:
# search_aggs
id_ = 4
suf = 'search'
actions_aggs[id_] = (
    search_history
    .filter(pl.col('action_type_id') == id_)
    .filter(pl.col('timestamp').dt.date() <= train_end_date)
    .filter(pl.col('timestamp').dt.date() >= train_end_date - timedelta(days=30 * 4))
    .group_by('user_id')
    .agg(
        pl.count('search_query').cast(pl.Int32).alias(f'num_{suf}'),
        pl.max('timestamp').alias(f'last_{suf}_time'),
        pl.min('timestamp').alias(f'first_{suf}_time'),
    )
    .with_columns([
        (pl.lit(val_start_date) - pl.col(f'last_{suf}_time'))
        .dt.total_days()
        .cast(pl.Int32)
        .alias(f'days_since_last_{suf}'),

        (pl.lit(val_start_date) - pl.col(f'first_{suf}_time'))
        .dt.total_days()
        .cast(pl.Int32)
        .alias(f'days_since_first_{suf}'),
    ])
    .select(
        'user_id',
        f'num_{suf}',
        f'days_since_last_{suf}',
        f'days_since_first_{suf}',
    )
)

In [13]:
actions_aggs.keys()

dict_keys([1, 2, 3, 5, 4])

In [14]:

def collect_if_lazy(df):
    if isinstance(df, pl.LazyFrame):
        return df.collect(streaming=True)
    return df

def downcast_float64_to_float32(df: pl.DataFrame) -> pl.DataFrame:
    float64_cols = [c for c, dt in zip(df.columns, df.dtypes) if dt == pl.Float64]
    if float64_cols:
        df = df.with_columns([pl.col(c).cast(pl.Float32) for c in float64_cols])
    return df

def build_hist_frames(
    actions_history,
    search_history,
    product_information,
    anchor_date,
    lookback_days: int = 120,
):
    history_end_date = anchor_date - timedelta(days=1)
    history_start_date = history_end_date - timedelta(days=lookback_days - 1)

    product_small = (
        product_information
        .select([
            'product_id',
            'price',
            'discount_price',
            'brand',
            'category_id',
        ])
    )

    act_hist = (
        actions_history
        .filter(pl.col('timestamp').dt.date() >= history_start_date)
        .filter(pl.col('timestamp').dt.date() <= history_end_date)
        .select([
            'user_id',
            'timestamp',
            'product_id',
            'page_product_id',
            'action_type_id',
            'widget_name_id',
        ])
        .join(product_small, on='product_id', how='left')
        .with_columns([
            pl.col('timestamp').dt.date().alias('event_date'),
            pl.when(pl.col('price').is_not_null() & (pl.col('price') > 0))
              .then((pl.col('price') - pl.col('discount_price')) / pl.col('price'))
              .otherwise(None)
              .cast(pl.Float32)
              .alias('discount_rate'),
            (
                pl.col('page_product_id').is_not_null() &
                (pl.col('page_product_id') != pl.col('product_id'))
            ).cast(pl.Int8).alias('is_cross_product_context'),
        ])
    )

    search_hist = (
        search_history
        .filter(pl.col('timestamp').dt.date() >= history_start_date)
        .filter(pl.col('timestamp').dt.date() <= history_end_date)
        .select([
            'user_id',
            'timestamp',
            'search_query',
            'action_type_id',
            'widget_name_id',
        ])
        .with_columns([
            pl.col('timestamp').dt.date().alias('event_date'),
            pl.col('search_query').fill_null('').str.len_chars().cast(pl.Int32).alias('query_len'),
        ])
    )

    return collect_if_lazy(act_hist), collect_if_lazy(search_hist)

def build_user_feature_table(users_df, anchor_date, actions_history, search_history, product_information):
    users_df = collect_if_lazy(users_df)

    act_hist, search_hist = build_hist_frames(
        actions_history=actions_history,
        search_history=search_history,
        product_information=product_information,
        anchor_date=anchor_date,
        lookback_days=120,
    )

    df = users_df

    feat = make_action_window_features(act_hist, anchor_date)
    df = df.join(feat, on='user_id', how='left')
    del feat
    gc.collect()
    
    print("Action window features added")

    feat = make_search_window_features(search_hist, anchor_date)
    df = df.join(feat, on='user_id', how='left')
    del feat
    gc.collect()
    
    print("Search window features added")

    feat = make_diversity_features(act_hist)
    df = df.join(feat, on='user_id', how='left')
    del feat
    gc.collect()
    
    print("Diversity features added")

    feat = make_concentration_features(act_hist)
    df = df.join(feat, on='user_id', how='left')
    del feat
    gc.collect()
    
    print("Concentration features added")

    feat = make_price_features(act_hist)
    df = df.join(feat, on='user_id', how='left')
    del feat
    gc.collect()
    
    print("Price features added")

    feat = make_context_features(act_hist)
    df = df.join(feat, on='user_id', how='left')
    del feat
    gc.collect()
    
    print("Context features added")

    feat = make_global_activity_features(act_hist, search_hist, anchor_date)
    df = df.join(feat, on='user_id', how='left')
    del feat
    gc.collect()
    
    print("Global activity features added")

    feat = make_search_text_features(search_hist)
    df = df.join(feat, on='user_id', how='left')
    del feat
    gc.collect()
    
    print("Sarch text features added")

    feat = make_preference_features_from_clicks(act_hist)
    df = df.join(feat, on='user_id', how='left')
    del feat
    gc.collect()
    
    print("Preference features added")

    df = add_trend_features(df)
    df = add_funnel_features(df)
    df = add_intensity_features(df)
    # df = add_search_alignment_features(df)

    df = downcast_float64_to_float32(df)

    del act_hist, search_hist
    gc.collect()

    return df

def sanitize_for_catboost(df_pd, num_cols, cat_cols, text_cols):
    for c in num_cols:
        if c in df_pd.columns:
            df_pd[c] = pd.to_numeric(df_pd[c], errors='coerce')
            if str(df_pd[c].dtype) == 'float64':
                df_pd[c] = df_pd[c].astype('float32')

    for c in cat_cols:
        if c in df_pd.columns:
            df_pd[c] = df_pd[c].astype('object')
            df_pd[c] = df_pd[c].where(df_pd[c].notna(), '__MISSING__')
            df_pd[c] = df_pd[c].astype(str)

    for c in text_cols:
        if c in df_pd.columns:
            df_pd[c] = df_pd[c].astype('object')
            df_pd[c] = df_pd[c].where(df_pd[c].notna(), '')
            df_pd[c] = df_pd[c].astype(str)

    return df_pd


In [15]:
def make_action_window_features(act_hist: pl.DataFrame, anchor_date):
    action_map = {
        1: 'click',
        2: 'favorite',
        3: 'order',
        5: 'to_cart',
    }
    windows = [7, 30, 90]

    exprs = []
    for action_id, suf in action_map.items():
        for w in windows:
            mask = (
                (pl.col('action_type_id') == action_id) &
                (pl.col('event_date') >= anchor_date - timedelta(days=w))
            )
            exprs.extend([
                pl.col('product_id').filter(mask).count().cast(pl.Int32).alias(f'num_{suf}_{w}d'),
                pl.col('event_date').filter(mask).n_unique().cast(pl.Int32).alias(f'active_days_{suf}_{w}d'),
            ])

    return act_hist.group_by('user_id').agg(exprs)

def make_search_window_features(search_hist: pl.DataFrame, anchor_date):
    windows = [7, 30, 90]
    exprs = []
    for w in windows:
        mask = pl.col('event_date') >= anchor_date - timedelta(days=w)
        exprs.extend([
            pl.col('search_query').filter(mask).count().cast(pl.Int32).alias(f'num_search_{w}d'),
            pl.col('event_date').filter(mask).n_unique().cast(pl.Int32).alias(f'active_days_search_{w}d'),
        ])
    return search_hist.group_by('user_id').agg(exprs)

In [16]:
def add_trend_features(df: pl.DataFrame) -> pl.DataFrame:
    def ratio(num_col, den_col, new_col):
        return (
            pl.when(pl.col(den_col).fill_null(0) > 0)
              .then(pl.col(num_col).fill_null(0) / pl.col(den_col).fill_null(0))
              .otherwise(0.0)
              .cast(pl.Float32)
              .alias(new_col)
        )

    return df.with_columns([
        ratio('num_click_7d', 'num_click_30d', 'click_trend_7_to_30'),
        ratio('num_to_cart_7d', 'num_to_cart_30d', 'to_cart_trend_7_to_30'),
        ratio('num_order_7d', 'num_order_30d', 'order_trend_7_to_30'),
        ratio('num_search_7d', 'num_search_30d', 'search_trend_7_to_30'),

        ratio('num_click_30d', 'num_click_90d', 'click_trend_30_to_90'),
        ratio('num_to_cart_30d', 'num_to_cart_90d', 'to_cart_trend_30_to_90'),
        ratio('num_order_30d', 'num_order_90d', 'order_trend_30_to_90'),
        ratio('num_search_30d', 'num_search_90d', 'search_trend_30_to_90'),
    ])

In [17]:
def add_funnel_features(df: pl.DataFrame) -> pl.DataFrame:
    def ratio(num_col, den_col, new_col):
        return (
            pl.when(pl.col(den_col).fill_null(0) > 0)
              .then(pl.col(num_col).fill_null(0) / pl.col(den_col).fill_null(0))
              .otherwise(0.0)
              .cast(pl.Float32)
              .alias(new_col)
        )

    return df.with_columns([
        ratio('num_to_cart_30d', 'num_click_30d', 'cart_per_click_30d'),
        ratio('num_order_30d', 'num_click_30d', 'order_per_click_30d'),
        ratio('num_order_30d', 'num_to_cart_30d', 'order_per_cart_30d'),
        ratio('num_favorite_30d', 'num_click_30d', 'favorite_per_click_30d'),
        ratio('num_order_30d', 'num_search_30d', 'order_per_search_30d'),
        ratio('num_to_cart_30d', 'num_search_30d', 'cart_per_search_30d'),

        ratio('num_to_cart_90d', 'num_click_90d', 'cart_per_click_90d'),
        ratio('num_order_90d', 'num_click_90d', 'order_per_click_90d'),
        ratio('num_order_90d', 'num_to_cart_90d', 'order_per_cart_90d'),
        ratio('num_order_90d', 'num_search_90d', 'order_per_search_90d'),
    ])

In [18]:
def add_intensity_features(df: pl.DataFrame) -> pl.DataFrame:
    def ratio(num_col, den_col, new_col):
        return (
            pl.when(pl.col(den_col).fill_null(0) > 0)
              .then(pl.col(num_col).fill_null(0) / pl.col(den_col).fill_null(0))
              .otherwise(0.0)
              .cast(pl.Float32)
              .alias(new_col)
        )

    return df.with_columns([
        ratio('num_click_30d', 'active_days_click_30d', 'clicks_per_active_day_30d'),
        ratio('num_to_cart_30d', 'active_days_to_cart_30d', 'to_cart_per_active_day_30d'),
        ratio('num_order_30d', 'active_days_order_30d', 'orders_per_active_day_30d'),
        ratio('num_search_30d', 'active_days_search_30d', 'searches_per_active_day_30d'),

        ratio('num_click_90d', 'active_days_click_90d', 'clicks_per_active_day_90d'),
        ratio('num_to_cart_90d', 'active_days_to_cart_90d', 'to_cart_per_active_day_90d'),
        ratio('num_order_90d', 'active_days_order_90d', 'orders_per_active_day_90d'),
        ratio('num_search_90d', 'active_days_search_90d', 'searches_per_active_day_90d'),
    ])

In [19]:
def make_diversity_features(act_hist: pl.DataFrame) -> pl.DataFrame:
    base = act_hist.group_by('user_id').agg([
        pl.col('product_id').filter(pl.col('action_type_id') == 1).n_unique().cast(pl.Int32).alias('unique_products_click_120d'),
        pl.col('product_id').filter(pl.col('action_type_id') == 3).n_unique().cast(pl.Int32).alias('unique_products_order_120d'),
        pl.col('product_id').filter(pl.col('action_type_id') == 5).n_unique().cast(pl.Int32).alias('unique_products_to_cart_120d'),

        pl.col('category_id').filter(pl.col('action_type_id') == 1).n_unique().cast(pl.Int32).alias('unique_categories_click_120d'),
        pl.col('category_id').filter(pl.col('action_type_id') == 3).n_unique().cast(pl.Int32).alias('unique_categories_order_120d'),
        pl.col('category_id').filter(pl.col('action_type_id') == 5).n_unique().cast(pl.Int32).alias('unique_categories_to_cart_120d'),

        pl.col('brand').filter(pl.col('action_type_id') == 1).n_unique().cast(pl.Int32).alias('unique_brands_click_120d'),
        pl.col('brand').filter(pl.col('action_type_id') == 3).n_unique().cast(pl.Int32).alias('unique_brands_order_120d'),
        pl.col('brand').filter(pl.col('action_type_id') == 5).n_unique().cast(pl.Int32).alias('unique_brands_to_cart_120d'),
    ])

    favorite_order_category = (
        act_hist
        .filter(pl.col('action_type_id') == 3)
        .group_by(['user_id', 'category_id'])
        .agg(pl.len().alias('cnt'))
        .sort(['user_id', 'cnt'], descending=[False, True])
        .group_by('user_id')
        .agg(pl.first('category_id').cast(pl.Utf8).alias('favorite_order_category_id'))
    )

    favorite_order_brand = (
        act_hist
        .filter(pl.col('action_type_id') == 3)
        .group_by(['user_id', 'brand'])
        .agg(pl.len().alias('cnt'))
        .sort(['user_id', 'cnt'], descending=[False, True])
        .group_by('user_id')
        .agg(pl.first('brand').cast(pl.Utf8).alias('favorite_order_brand'))
    )

    return (
        base
        .join(favorite_order_category, on='user_id', how='left')
        .join(favorite_order_brand, on='user_id', how='left')
    )

In [20]:
def make_concentration_features(act_hist: pl.DataFrame) -> pl.DataFrame:
    order_total = (
        act_hist
        .filter(pl.col('action_type_id') == 3)
        .group_by('user_id')
        .agg(pl.len().alias('num_order_events_120d'))
    )

    order_cat_top = (
        act_hist
        .filter(pl.col('action_type_id') == 3)
        .group_by(['user_id', 'category_id'])
        .agg(pl.len().alias('cnt'))
        .sort(['user_id', 'cnt'], descending=[False, True])
        .group_by('user_id')
        .agg(pl.first('cnt').alias('top_category_order_cnt'))
    )

    order_brand_top = (
        act_hist
        .filter(pl.col('action_type_id') == 3)
        .group_by(['user_id', 'brand'])
        .agg(pl.len().alias('cnt'))
        .sort(['user_id', 'cnt'], descending=[False, True])
        .group_by('user_id')
        .agg(pl.first('cnt').alias('top_brand_order_cnt'))
    )

    click_total = (
        act_hist
        .filter(pl.col('action_type_id') == 1)
        .group_by('user_id')
        .agg(pl.len().alias('num_click_events_120d'))
    )

    click_cat_top = (
        act_hist
        .filter(pl.col('action_type_id') == 1)
        .group_by(['user_id', 'category_id'])
        .agg(pl.len().alias('cnt'))
        .sort(['user_id', 'cnt'], descending=[False, True])
        .group_by('user_id')
        .agg(pl.first('cnt').alias('top_category_click_cnt'))
    )

    return (
        order_total
        .join(order_cat_top, on='user_id', how='left')
        .join(order_brand_top, on='user_id', how='left')
        .join(click_total, on='user_id', how='left')
        .join(click_cat_top, on='user_id', how='left')
        .with_columns([
            pl.when(pl.col('num_order_events_120d') > 0)
              .then(pl.col('top_category_order_cnt') / pl.col('num_order_events_120d'))
              .otherwise(0.0)
              .cast(pl.Float32)
              .alias('top_category_order_share_120d'),

            pl.when(pl.col('num_order_events_120d') > 0)
              .then(pl.col('top_brand_order_cnt') / pl.col('num_order_events_120d'))
              .otherwise(0.0)
              .cast(pl.Float32)
              .alias('top_brand_order_share_120d'),

            pl.when(pl.col('num_click_events_120d') > 0)
              .then(pl.col('top_category_click_cnt') / pl.col('num_click_events_120d'))
              .otherwise(0.0)
              .cast(pl.Float32)
              .alias('top_category_click_share_120d'),
        ])
        .select(
            'user_id',
            'top_category_order_share_120d',
            'top_brand_order_share_120d',
            'top_category_click_share_120d',
        )
    )

In [21]:
def make_price_features(act_hist: pl.DataFrame) -> pl.DataFrame:
    return act_hist.group_by('user_id').agg([
        pl.col('discount_price').filter(pl.col('action_type_id') == 1).mean().cast(pl.Float32).alias('avg_discount_price_click_120d'),
        pl.col('discount_price').filter(pl.col('action_type_id') == 1).median().cast(pl.Float32).alias('median_discount_price_click_120d'),
        pl.col('discount_price').filter(pl.col('action_type_id') == 1).std().cast(pl.Float32).alias('std_discount_price_click_120d'),

        pl.col('discount_price').filter(pl.col('action_type_id') == 5).mean().cast(pl.Float32).alias('avg_discount_price_to_cart_120d'),
        pl.col('discount_price').filter(pl.col('action_type_id') == 5).median().cast(pl.Float32).alias('median_discount_price_to_cart_120d'),
        pl.col('discount_price').filter(pl.col('action_type_id') == 5).std().cast(pl.Float32).alias('std_discount_price_to_cart_120d'),

        pl.col('discount_price').filter(pl.col('action_type_id') == 3).mean().cast(pl.Float32).alias('avg_discount_price_order_120d'),
        pl.col('discount_price').filter(pl.col('action_type_id') == 3).median().cast(pl.Float32).alias('median_discount_price_order_120d'),
        pl.col('discount_price').filter(pl.col('action_type_id') == 3).std().cast(pl.Float32).alias('std_discount_price_order_120d'),

        pl.col('discount_rate').filter(pl.col('action_type_id') == 1).mean().cast(pl.Float32).alias('avg_discount_rate_click_120d'),
        pl.col('discount_rate').filter(pl.col('action_type_id') == 5).mean().cast(pl.Float32).alias('avg_discount_rate_to_cart_120d'),
        pl.col('discount_rate').filter(pl.col('action_type_id') == 3).mean().cast(pl.Float32).alias('avg_discount_rate_order_120d'),
    ])

In [22]:
def make_context_features(act_hist: pl.DataFrame) -> pl.DataFrame:
    return act_hist.group_by('user_id').agg([
        pl.col('is_cross_product_context').filter(pl.col('action_type_id') == 1).mean().cast(pl.Float32).alias('cross_product_click_share_120d'),
        pl.col('is_cross_product_context').filter(pl.col('action_type_id') == 5).mean().cast(pl.Float32).alias('cross_product_to_cart_share_120d'),

        pl.col('widget_name_id').filter(pl.col('action_type_id') == 1).n_unique().cast(pl.Int32).alias('unique_widgets_click_120d'),
        pl.col('widget_name_id').filter(pl.col('action_type_id') == 3).n_unique().cast(pl.Int32).alias('unique_widgets_order_120d'),

        pl.col('widget_name_id').filter(pl.col('action_type_id') == 1).mode().first().cast(pl.Utf8).alias('top_widget_click'),
        pl.col('widget_name_id').filter(pl.col('action_type_id') == 3).mode().first().cast(pl.Utf8).alias('top_widget_order'),
    ])

In [23]:
def make_global_activity_features(act_hist: pl.DataFrame, search_hist: pl.DataFrame, anchor_date) -> pl.DataFrame:
    act_any = act_hist.select('user_id', 'event_date')
    search_any = search_hist.select('user_id', 'event_date')

    union_hist = pl.concat([act_any, search_any])

    return union_hist.group_by('user_id').agg([
        pl.len().cast(pl.Int32).alias('total_activity_120d'),
        pl.col('event_date').n_unique().cast(pl.Int32).alias('active_days_any_120d'),
        (pl.lit(anchor_date) - pl.col('event_date').max()).dt.total_days().cast(pl.Int32).alias('days_since_last_any_activity'),
    ])

In [24]:
def make_search_text_features(search_hist: pl.DataFrame) -> pl.DataFrame:
    return (
        search_hist
        .sort(['user_id', 'timestamp'])
        .group_by('user_id')
        .agg([
            pl.len().cast(pl.Int32).alias('num_search_events_120d'),
            pl.col('search_query').n_unique().cast(pl.Int32).alias('unique_search_queries_120d'),
            pl.col('query_len').mean().cast(pl.Float32).alias('avg_query_len_120d'),
            pl.col('search_query').last().cast(pl.Utf8).alias('last_search_query'),
            pl.col('search_query').tail(10).alias('recent_queries_list'),
        ])
        .with_columns([
            pl.when(pl.col('num_search_events_120d') > 0)
              .then(1.0 - pl.col('unique_search_queries_120d') / pl.col('num_search_events_120d'))
              .otherwise(0.0)
              .cast(pl.Float32)
              .alias('repeat_search_share_120d'),

            pl.col('recent_queries_list').list.join(' [SEP] ').alias('recent_queries_text_120d'),
        ])
        .drop('recent_queries_list')
    )

In [25]:
def add_search_alignment_features(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns([
        (pl.col('days_since_last_click').fill_null(999) - pl.col('days_since_last_search').fill_null(999))
            .cast(pl.Int32)
            .alias('search_to_click_gap'),

        (pl.col('days_since_last_order').fill_null(999) - pl.col('days_since_last_search').fill_null(999))
            .cast(pl.Int32)
            .alias('search_to_order_gap'),

        pl.when(pl.col('num_order_30d').fill_null(0) > 0)
          .then(pl.col('num_search_30d').fill_null(0) / pl.col('num_order_30d').fill_null(0))
          .otherwise(pl.col('num_search_30d').fill_null(0))
          .cast(pl.Float32)
          .alias('search_without_order_30d'),
    ])

In [26]:
def make_preference_features_from_clicks(act_hist: pl.DataFrame) -> pl.DataFrame:
    favorite_click_category = (
        act_hist
        .filter(pl.col('action_type_id') == 1)
        .group_by(['user_id', 'category_id'])
        .agg(pl.len().alias('cnt'))
        .sort(['user_id', 'cnt'], descending=[False, True])
        .group_by('user_id')
        .agg(pl.first('category_id').cast(pl.Utf8).alias('favorite_click_category_id'))
    )

    favorite_click_brand = (
        act_hist
        .filter(pl.col('action_type_id') == 1)
        .group_by(['user_id', 'brand'])
        .agg(pl.len().alias('cnt'))
        .sort(['user_id', 'cnt'], descending=[False, True])
        .group_by('user_id')
        .agg(pl.first('brand').cast(pl.Utf8).alias('favorite_click_brand'))
    )

    return favorite_click_category.join(favorite_click_brand, on='user_id', how='left')

In [ ]:

df_extra = build_user_feature_table(
    users_df=val_target,
    anchor_date=val_start_date,
    actions_history=actions_history,
    search_history=search_history,
    product_information=product_information,
)

df_extra.shape


/tmp/ipykernel_40740/947966847.py:3: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  return df.collect(streaming=True)


In [ ]:
# replaced by build_user_feature_table(...)

In [ ]:
# replaced by build_user_feature_table(...)

In [ ]:
# replaced by build_user_feature_table(...)

In [ ]:
# replaced by build_user_feature_table(...)

In [ ]:
# replaced by build_user_feature_table(...)

In [ ]:
# replaced by build_user_feature_table(...)

In [ ]:
# replaced by build_user_feature_table(...)

In [ ]:
# replaced by build_user_feature_table(...)

In [ ]:
# replaced by build_user_feature_table(...)

In [ ]:
# replaced by build_user_feature_table(...)

In [ ]:
# replaced by build_user_feature_table(...)

In [ ]:
# replaced by build_user_feature_table(...)

In [ ]:
# replaced by build_user_feature_table(...)

In [ ]:
# replaced by build_user_feature_table(...)

ColumnNotFoundError: unable to find column "days_since_last_click"; valid columns: ["user_id", "target", "num_click_7d", "active_days_click_7d", "num_click_30d", "active_days_click_30d", "num_click_90d", "active_days_click_90d", "num_favorite_7d", "active_days_favorite_7d", "num_favorite_30d", "active_days_favorite_30d", "num_favorite_90d", "active_days_favorite_90d", "num_order_7d", "active_days_order_7d", "num_order_30d", "active_days_order_30d", "num_order_90d", "active_days_order_90d", "num_to_cart_7d", "active_days_to_cart_7d", "num_to_cart_30d", "active_days_to_cart_30d", "num_to_cart_90d", "active_days_to_cart_90d", "num_search_7d", "active_days_search_7d", "num_search_30d", "active_days_search_30d", "num_search_90d", "active_days_search_90d", "unique_products_click_120d", "unique_products_order_120d", "unique_products_to_cart_120d", "unique_categories_click_120d", "unique_categories_order_120d", "unique_categories_to_cart_120d", "unique_brands_click_120d", "unique_brands_order_

In [ ]:

cat_cols = [
    'favorite_order_category_id',
    'favorite_order_brand',
    'favorite_click_category_id',
    'favorite_click_brand',
    'top_widget_click',
    'top_widget_order',
]

text_cols = [
    'last_search_query',
    'recent_queries_text_120d',
]

num_cols = [
    c for c in df_extra.columns
    if c not in ['user_id', 'target'] + cat_cols + text_cols
]

all_cols = num_cols + cat_cols + text_cols

train_pl = df_extra.filter((pl.col('user_id') % 10) <= 6)
eval_pl = df_extra.filter((pl.col('user_id') % 10) > 6)

del df_extra
gc.collect()

train_df = sanitize_for_catboost(train_pl.to_pandas(), num_cols, cat_cols, text_cols)
eval_df = sanitize_for_catboost(eval_pl.to_pandas(), num_cols, cat_cols, text_cols)

del train_pl, eval_pl
gc.collect()

train_y = train_df['target'].reset_index(drop=True)
eval_y = eval_df['target'].reset_index(drop=True)

train_df = train_df[all_cols].reset_index(drop=True)
eval_df = eval_df[all_cols].reset_index(drop=True)


In [ ]:
train_df.shape, eval_df.shape

In [ ]:
len(all_cols), len(cat_cols), len(text_cols)

Index(['user_id', 'target', 'num_products_click', 'sum_discount_price_click',
       'max_discount_price_click', 'days_since_last_click',
       'days_since_first_click', 'num_products_favorite',
       'sum_discount_price_favorite', 'max_discount_price_favorite',
       'days_since_last_favorite', 'days_since_first_favorite',
       'num_products_order', 'sum_discount_price_order',
       'max_discount_price_order', 'days_since_last_order',
       'days_since_first_order', 'num_products_to_cart',
       'sum_discount_price_to_cart', 'max_discount_price_to_cart',
       'days_since_last_to_cart', 'days_since_first_to_cart', 'num_search',
       'days_since_last_search', 'days_since_first_search'],
      dtype='object')

In [ ]:
all_cols[:20]

In [ ]:
# train_df / eval_df are already sanitized and aligned above

In [ ]:
train_pool = catboost.Pool(
    train_df,
    label=train_y,
    cat_features=cat_cols,
    text_features=text_cols,
)

eval_pool = catboost.Pool(
    eval_df,
    label=eval_y,
    cat_features=cat_cols,
    text_features=text_cols,
)

del train_df, eval_df, train_y, eval_y
gc.collect()


In [ ]:
train_pool.shape, eval_pool.shape

((1311636, 99), (563320, 99))

In [ ]:
params = {
    'iterations': 2000,
    'depth': 7,
    'learning_rate': 0.05,
    'random_state': 1,
    'eval_metric': 'AUC',
    'loss_function': 'Logloss',
    'task_type': 'GPU',
    'od_type': 'Iter',
    'od_wait': 50,
}

In [ ]:
model = catboost.CatBoostClassifier(**params)
model.fit(
    train_pool,
    eval_set=eval_pool,
    use_best_model=True,
    verbose=50,
)

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7418671	best: 0.7418671 (0)	total: 1.9s	remaining: 1h 3m 17s
50:	test: 0.7582741	best: 0.7582741 (50)	total: 5.45s	remaining: 3m 28s
100:	test: 0.7598111	best: 0.7598111 (100)	total: 8.53s	remaining: 2m 40s
150:	test: 0.7605194	best: 0.7605194 (150)	total: 11.4s	remaining: 2m 19s
200:	test: 0.7611210	best: 0.7611210 (200)	total: 14s	remaining: 2m 5s
250:	test: 0.7614433	best: 0.7614433 (250)	total: 16.5s	remaining: 1m 54s
300:	test: 0.7617009	best: 0.7617009 (300)	total: 18.7s	remaining: 1m 45s
350:	test: 0.7619208	best: 0.7619208 (350)	total: 20.9s	remaining: 1m 38s
400:	test: 0.7620431	best: 0.7620507 (384)	total: 23.1s	remaining: 1m 32s
450:	test: 0.7621564	best: 0.7621593 (448)	total: 25.2s	remaining: 1m 26s
500:	test: 0.7622930	best: 0.7622956 (499)	total: 27.3s	remaining: 1m 21s
550:	test: 0.7623813	best: 0.7624051 (541)	total: 29.5s	remaining: 1m 17s
600:	test: 0.7624752	best: 0.7624752 (600)	total: 31.5s	remaining: 1m 13s
650:	test: 0.7625298	best: 0.7625400 (647)	to

In [ ]:
print("Best iteration:", model.best_iteration_)
print("Best score:", model.best_score_)
print("Trees in final model:", model.tree_count_)

Best iteration: 901
Best score: {'learn': {'Logloss': 0.5136070049159981}, 'validation': {'Logloss': 0.5169792480295392, 'AUC': 0.7628807127475739}}
Trees in final model: 902


In [ ]:
name = 'baseline1'
model.save_model(f"{name}.bin")

In [ ]:
fi = model.get_feature_importance(eval_pool, prettified=True)
fi.head(50)

,Feature Id,Importances
0,active_days_order_30d,13.062981
1,active_days_order_90d,9.463212
2,recent_queries_text_120d,9.243945
3,num_order_90d,3.274506
4,active_days_search_90d,3.006503
5,cart_per_click_90d,2.997617
6,avg_discount_price_to_cart_120d,2.594021
7,orders_per_active_day_90d,2.418554
8,active_days_to_cart_90d,2.249304
9,active_days_order_7d,1.989399


In [ ]:

test_users_submission = pl.read_csv(os.path.join(data_path, 'test_users.csv'))

df_test = build_user_feature_table(
    users_df=test_users_submission,
    anchor_date=test_start_date,
    actions_history=actions_history,
    search_history=search_history,
    product_information=product_information,
)

df_test.shape


In [ ]:

df_test_pd = sanitize_for_catboost(df_test.to_pandas(), num_cols, cat_cols, text_cols)

for c in all_cols:
    if c not in df_test_pd.columns:
        if c in num_cols:
            df_test_pd[c] = np.nan
        elif c in cat_cols:
            df_test_pd[c] = '__MISSING__'
        elif c in text_cols:
            df_test_pd[c] = ''

df_test_pd = df_test_pd[['user_id'] + all_cols].copy()

del df_test
gc.collect()

df_test_pd.shape


NameError: name 'action_window_features_test' is not defined

In [ ]:

test_pool = catboost.Pool(
    df_test_pd[all_cols],
    cat_features=cat_cols,
    text_features=text_cols,
)


In [ ]:

df_test_pd['predict'] = model.predict(
    test_pool,
    prediction_type='Probability'
)[:, 1]

df_test_pd[['user_id', 'predict']].head()


(datetime.date(2024, 7, 31), datetime.date(2024, 4, 2))

In [ ]:

submission = df_test_pd[['user_id', 'predict']].copy()
submission.to_csv('baseline1_submission.csv', index=False)

del test_pool
gc.collect()

submission.head()


In [ ]:
# obsolete baseline test pipeline removed

In [ ]:
# obsolete baseline test pipeline removed

In [ ]:
# obsolete baseline test pipeline removed

In [ ]:
# obsolete baseline test pipeline removed

(2068424, 24)

In [ ]:
# obsolete baseline test pipeline removed

KeyError: "None of [Index(['num_click_7d', 'active_days_click_7d', 'num_click_30d',\n       'active_days_click_30d', 'num_click_90d', 'active_days_click_90d',\n       'num_favorite_7d', 'active_days_favorite_7d', 'num_favorite_30d',\n       'active_days_favorite_30d', 'num_favorite_90d',\n       'active_days_favorite_90d', 'num_order_7d', 'active_days_order_7d',\n       'num_order_30d', 'active_days_order_30d', 'num_order_90d',\n       'active_days_order_90d', 'num_to_cart_7d', 'active_days_to_cart_7d',\n       'num_to_cart_30d', 'active_days_to_cart_30d', 'num_to_cart_90d',\n       'active_days_to_cart_90d', 'num_search_7d', 'active_days_search_7d',\n       'num_search_30d', 'active_days_search_30d', 'num_search_90d',\n       'active_days_search_90d', 'unique_products_click_120d',\n       'unique_products_order_120d', 'unique_products_to_cart_120d',\n       'unique_categories_click_120d', 'unique_categories_order_120d',\n       'unique_categories_to_cart_120d', 'unique_brands_click_12

In [ ]:
# obsolete baseline test pipeline removed

,user_id,predict
0,1342,0.178745
1,9852,0.809109
2,10206,0.222153
3,11317,0.221137
4,13289,0.611800
...,...,...
2068419,11157283,0.222715
2068420,11160395,0.141347
2068421,11165052,0.602636
2068422,11168218,0.513334


In [ ]:
# obsolete baseline test pipeline removed